# RadioML MLP Inference — PYNQ-Z2

**Board:** PYNQ-Z2 (Zynq XC7Z020)  
**Model:** 4-layer MLP  `256 → 512 → 256 → 128 → 11`  
**Task:** RadioML 2016.10a modulation classification (11 classes)  
**Classes:** BPSK, QPSK, 8PSK, 16QAM, 64QAM, BFSK, CPFSK, PAM4, WB-FM, AM-SSB, AM-DSB

---

### Files required in this folder

| File | Source |
|------|--------|
| `mlp_accelerator.bit` | Vivado output → `fpga/vivado/create_project.tcl` |
| `mlp_accelerator.hwh` | Extracted from `mlp_accelerator.xsa` via `fpga/pynq/extract_hwh.py` |

> **Note:** Weights are baked into BRAM at bitstream generation time — no `mlp_weights.bin` needed on the board.

### Stream protocol (ap_ctrl_none — no hls_ip control needed)
- **Send:** 256 uint32 words — one Q4.12 int16 feature in bits[15:0] per word (1024 bytes total)
- **Recv:** 1 uint32 word — argmax class index 0..10 (4 bytes total)
- **Usage:** `dma.sendchannel.transfer(in_buf)` → `dma.recvchannel.transfer(out_buf)` → `wait()` × 2

No `hls_ip.write(0x00, 0x01)` required — the kernel runs as a free-running streaming pipeline.


In [ ]:
# ============================================================
# CELL 1: Imports & Constants
# ============================================================
import numpy as np
import time
import io
import os
from pynq import Overlay, allocate
import ipywidgets as widgets
from IPython.display import display

# -- Model-specific constants (must match mlp_top.h and train_radioml.py) --
N_IN   = 256        # input features: 2 channels x 128 IQ samples
N_OUT  = 11         # output classes
SCALE  = 4096.0     # Q4.12: 1 << 12

CLASSES = [
    'BPSK', 'QPSK', '8PSK', '16QAM', '64QAM',
    'BFSK', 'CPFSK', 'PAM4', 'WB-FM', 'AM-SSB', 'AM-DSB',
]

BIT_FILE = 'mlp_accelerator.bit'   # weights baked into BRAM -- no .bin needed

print("Constants defined.")
print(f"  Input size     : {N_IN}")
print(f"  Output classes : {N_OUT}  -> {CLASSES}")
print(f"  Fixed-point    : Q4.12  (SCALE = {int(SCALE)})")
print(f"  Bitstream      : {BIT_FILE}")


In [ ]:
# ============================================================
# CELL 2: Load Bitstream onto FPGA
# ============================================================
if not os.path.exists(BIT_FILE):
    raise FileNotFoundError(
        f"Required file not found: {BIT_FILE}\n"
        "  Build with: vitis_hls -f fpga/hls/run_hls.tcl\n"
        "  Then:       vivado -mode batch -source fpga/vivado/create_project.tcl"
    )

print(f"Loading bitstream: {BIT_FILE} ...")
overlay = Overlay(BIT_FILE)
print("Bitstream loaded successfully!")
print("\nAvailable IP blocks:", list(overlay.ip_dict.keys()))


In [ ]:
# ============================================================
# CELL 3: Acquire DMA Handle
# ============================================================
# ap_ctrl_none: the HLS kernel has NO AXI-Lite control port.
# Only the DMA is needed -- no hls_ip handle required.
dma = overlay.axi_dma_0

print("DMA handle:", dma)
print("Kernel is free-running -- no ap_start write needed.")


In [ ]:
# ============================================================
# CELL 4: Preprocessing & Inference Function
# ============================================================

def preprocess(x):
    """
    Normalise IQ frame and quantise to Q4.12 uint32 words.

    Accepts:
      - shape (256,)   flat [I x128, Q x128]
      - shape (128, 2) time-domain rows [I, Q]
      - shape (2, 128) channel-first

    Returns: uint32 ndarray of length 256
      Each element = int16 Q4.12 value in bits[15:0], bits[31:16] = 0.
    """
    x = np.asarray(x, dtype=np.float32)
    if x.shape == (128, 2):
        x = x.T.flatten()
    elif x.shape == (2, 128):
        x = x.flatten()
    else:
        x = x.flatten()
        if len(x) > N_IN:
            x = x[:N_IN]
        elif len(x) < N_IN:
            x = np.pad(x, (0, N_IN - len(x)))

    mx = np.abs(x).max()
    if mx > 1e-8:
        x = x / mx

    q = np.clip(np.round(x * SCALE), -32768, 32767).astype(np.int16)
    return q.astype(np.uint16).astype(np.uint32)


def run_inference(x):
    """
    Run one forward pass through the FPGA MLP accelerator.

    Parameters
    ----------
    x : array-like, shape (256,) -- IQ input frame (float32)

    Returns
    -------
    pred : int -- predicted class index (0..10)
    ms   : float -- DMA round-trip latency in ms
    """
    in_buf  = allocate(shape=(N_IN,), dtype=np.uint32)
    out_buf = allocate(shape=(1,),    dtype=np.uint32)

    np.copyto(in_buf, preprocess(x))

    t0 = time.perf_counter()
    dma.sendchannel.transfer(in_buf)
    dma.recvchannel.transfer(out_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    t1 = time.perf_counter()

    pred = int(out_buf[0])

    in_buf.freebuffer()
    out_buf.freebuffer()

    return pred, (t1 - t0) * 1000.0


print("Preprocessing and inference functions ready.")


In [ ]:
# ============================================================
# CELL 5: Single-Sample Inference Test (Random Input)
# ============================================================
print("=" * 55)
print("  Single Inference Test -- Random IQ Frame")
print("=" * 55)

x_test = np.random.uniform(-1.0, 1.0, N_IN).astype(np.float32)

pred, ms = run_inference(x_test)

class_name = CLASSES[pred] if 0 <= pred < len(CLASSES) else f"Unknown({pred})"

print(f"Prediction  : Class {pred}  ({class_name})")
print(f"Latency     : {ms:.3f} ms  (end-to-end incl. DMA)")
print()
print("(Random input -- prediction has no ground-truth meaning here.)")


In [ ]:
# ============================================================
# CELL 6: Interactive Upload -- .npy or .csv IQ Samples
# ============================================================
# Upload format:
#   .npy  -- numpy array, shape (N, 256), dtype float32
#   .csv  -- N rows, 256 columns of float values
# Each row = one IQ frame [I(128 samples), Q(128 samples)] normalised to [-1, 1]

print("=" * 60)
print("  RadioML MLP FPGA Classifier -- Upload IQ Samples")
print("=" * 60)

upload_btn = widgets.FileUpload(
    accept='.npy,.csv',
    multiple=False,
    description='Upload File',
    layout=widgets.Layout(width='180px')
)
out_widget = widgets.Output()

def on_upload_change(change):
    with out_widget:
        out_widget.clear_output()

        uploaded = list(upload_btn.value.values())[0]
        content  = uploaded['content']
        fname    = uploaded['name']

        try:
            if fname.endswith('.npy'):
                data = np.load(io.BytesIO(content))
            elif fname.endswith('.csv'):
                data = np.genfromtxt(io.BytesIO(content), delimiter=',')
            else:
                print(f"Unsupported format: {fname}  (use .npy or .csv)")
                return

            data = np.asarray(data, dtype=np.float32)
            if data.ndim == 1:
                data = data.reshape(1, -1)

            n_samples = len(data)
            print(f"Loaded {n_samples} samples from '{fname}'")
            print("-" * 55)
            print(f"{'#':<5} | {'Prediction':<12} | {'Latency':>8}")
            print("-" * 55)

            t_total     = 0.0
            pred_counts = {}

            for i in range(n_samples):
                pred, ms = run_inference(data[i])
                t_total += ms

                class_name = CLASSES[pred] if 0 <= pred < len(CLASSES) else f"Unknown({pred})"
                pred_counts[class_name] = pred_counts.get(class_name, 0) + 1

                print(f"{i+1:<5} | {class_name:<12} | {ms:7.3f} ms")

            print("-" * 55)
            print(f"Total: {n_samples} samples  |  Avg: {t_total/n_samples:.3f} ms per sample")
            print()
            print("Classification summary:")
            for cls in CLASSES:
                cnt = pred_counts.get(cls, 0)
                pct = 100.0 * cnt / n_samples
                bar = '#' * int(pct / 2)
                print(f"  {cls:<10} : {cnt:4d} ({pct:5.1f}%)  {bar}")

        except Exception as e:
            print(f"Error: {e}")
            import traceback
            traceback.print_exc()

        upload_btn.value.clear()

upload_btn.observe(on_upload_change, names='value')

display(
    widgets.HTML(
        "<h3>RadioML MLP Hardware Classifier</h3>"
        "<p>Upload a <b>.npy</b> (shape: N x 256, float32) "
        "or <b>.csv</b> (256 columns) file of IQ frames.<br>"
        "The FPGA MLP classifies each sample as one of 11 modulation types.</p>"
    ),
    upload_btn,
    out_widget
)


In [ ]:
# ============================================================
# CELL 7: Batch Throughput Benchmark
# ============================================================
print("=" * 60)
print("  Batch Benchmark -- 100 Random IQ Frames")
print("=" * 60)

N_BENCH = 100
xs      = np.random.uniform(-1.0, 1.0, (N_BENCH, N_IN)).astype(np.float32)

latencies   = []
predictions = []

t_wall_start = time.perf_counter()
for i in range(N_BENCH):
    pred, ms = run_inference(xs[i])
    latencies.append(ms)
    predictions.append(pred)
t_wall_end = time.perf_counter()

latencies = np.array(latencies)
total_ms  = (t_wall_end - t_wall_start) * 1000.0

print(f"Samples Processed  : {N_BENCH}")
print(f"Pure HW Latency    : ~0.05 ms  (from HLS synthesis estimate)")
print(f"Avg System Latency : {latencies.mean():.3f} ms  (incl. DMA overhead)")
print(f"Min / Max          : {latencies.min():.3f} ms / {latencies.max():.3f} ms")
print(f"Throughput         : {1000.0 * N_BENCH / total_ms:.0f} inferences/sec")
print()

from collections import Counter
counts = Counter(CLASSES[p] if 0 <= p < len(CLASSES) else f"Unknown({p})" for p in predictions)
print("Prediction distribution (random input -- roughly uniform expected):")
for cls in CLASSES:
    cnt = counts.get(cls, 0)
    bar = '#' * cnt
    print(f"  {cls:<10} : {cnt:3d}  {bar}")


In [ ]:
# ============================================================
# CELL 8: Debug -- DMA & Overlay Inspection
# ============================================================
# ap_ctrl_none: NO AXI-Lite register map on mlp_top_0 -- hls_ip does not exist.
# Use this cell to verify the DMA is accessible and the overlay loaded correctly.

print("Overlay IP dict:")
for name, info in overlay.ip_dict.items():
    print(f"  {name:<30} : {info.get('type', '?')}")

print()
print("DMA object:", dma)
print()
print("DMA channel status:")
try:
    print(f"  sendchannel  : running={dma.sendchannel.running}")
    print(f"  recvchannel  : running={dma.recvchannel.running}")
except Exception as e:
    print(f"  (could not read channel status: {e})")

print()
print("If inference hangs: reload the overlay with Overlay(BIT_FILE)")
print("  overlay = Overlay('mlp_accelerator.bit')")
print("  dma = overlay.axi_dma_0")
